In [4]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = SparkSession.builder.appName("Lab2-Transactions").getOrCreate()
spark.sparkContext.setLogLevel("WARN")
df = spark.read.json("transactions_10k.jsonl")
df = df.withColumn("timestamp", to_timestamp(col("timestamp"), "yyyy-MM-dd HH:mm:ss"))

print(f"Sukces! Liczba rekordów: {df.count()}")
df.show(5, truncate=False)

Sukces! Liczba rekordów: 10000
+------+-----------+--------+-------------------+-------+-------+
|amount|category   |store   |timestamp          |tx_id  |user_id|
+------+-----------+--------+-------------------+-------+-------+
|312.32|elektronika|Warszawa|2026-04-12 08:25:07|TX00001|u48    |
|79.57 |książki    |Warszawa|2026-04-12 08:05:43|TX00002|u15    |
|126.17|odzież     |Warszawa|2026-04-12 09:15:30|TX00003|u18    |
|34.08 |odzież     |Warszawa|2026-04-12 10:05:39|TX00004|u10    |
|428.88|żywność    |Kraków  |2026-04-12 09:04:36|TX00005|u17    |
+------+-----------+--------+-------------------+-------+-------+
only showing top 5 rows



In [5]:
from pyspark.sql.functions import avg, col, desc, lit, round as _round, window

(
    df.filter(col("store") == "Gdańsk")
    .groupBy(window("timestamp", "1 hour"))
    .agg(_round(avg("amount"), 2).alias("avg_amount"))
    .orderBy(col("avg_amount").asc())
    .select(
        col("window.start").alias("godz_od"),
        col("window.end").alias("godz_do"),
        "avg_amount",
    )
    .show(1, truncate=False)
)

t0 = lit("2024-03-29 09:00:00").cast("timestamp")
t1 = lit("2024-03-29 09:30:00").cast("timestamp")
(
    df.filter((col("timestamp") >= t0) & (col("timestamp") < t1))
    .groupBy("category")
    .count()
    .orderBy("category")
    .show()
)

(
    df.groupBy(window("timestamp", "15 minutes"))
    .count()
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        col("count").alias("liczba_tx"),
    )
    .orderBy(desc("liczba_tx"))
    .show(1, truncate=False)
)

+-------------------+-------------------+----------+
|godz_od            |godz_do            |avg_amount|
+-------------------+-------------------+----------+
|2026-04-12 08:00:00|2026-04-12 09:00:00|395.01    |
+-------------------+-------------------+----------+
only showing top 1 row

+--------+-----+
|category|count|
+--------+-----+
+--------+-----+

+-------------------+-------------------+---------+
|od                 |do                 |liczba_tx|
+-------------------+-------------------+---------+
|2026-04-12 09:15:00|2026-04-12 09:30:00|1234     |
+-------------------+-------------------+---------+
only showing top 1 row

